# Solving the 1D Poisson Equation with PyPIELM

This notebook demonstrates the complete **PyPIELM** workflow for a simple but representative PDE:

$$-u''(x) = \pi^2 \sin(\pi x), \quad x \in [0, 1], \quad u(0) = u(1) = 0$$

**Exact solution:** $u(x) = \sin(\pi x)$

We will:
1. Set up a `PIELMDataset` with collocation points and boundary conditions
2. Define a PDE operator that encodes the physics
3. Fit `CorePIELM` — an analytic, matrix-free solver (no gradient descent!)
4. Visualise the solution and pointwise error using `pypielm.visualization`
5. Compare several model variants

In [1]:
import math
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove if running interactively
import matplotlib.pyplot as plt

from pypielm.data.dataset import PIELMDataset
from pypielm.models import CorePIELM, GFFPIELM, BayesianPIELM
from pypielm.core.solver import WeightedLinearSystem
from pypielm.visualization import (
    plot_solution_1d,
    plot_pareto,
    save_figure,
)

SEED = 42
rng  = np.random.default_rng(SEED)

## 1. Build the Dataset

`PIELMDataset.from_arrays` accepts:
- `X_colloc` — interior collocation points where the PDE residual is enforced
- `X_bc`, `y_bc` — boundary condition points and values
- Optionally `X_data`, `y_data` — observed solution values (data-fitting)

In [2]:
N_COLLOC = 300

# Interior collocation points drawn uniformly
X_c = rng.uniform(0.0, 1.0, (N_COLLOC, 1))

# Dirichlet boundary conditions: u(0)=0, u(1)=0
X_bc = np.array([[0.0], [1.0]])
y_bc = np.array([0.0, 0.0])

dataset = PIELMDataset.from_arrays(
    X_c,
    X_bc=X_bc, y_bc=y_bc,
)
print(dataset)

PIELMDataset(colloc=300, bc=2, ic=0, obs=0)


## 2. Define the PDE Operator

The PDE $-u'' = \pi^2 \sin(\pi x)$ maps to a block equation on the feature matrix $H$:

$$-H''\,\beta = \pi^2 \sin(\pi x) \quad \Longleftrightarrow \quad \tilde{H}\,\beta = \tilde{y}$$

where $\tilde{H} = -H''$ and $\tilde{y} = \pi^2 \sin(\pi x)$.

The PDE operator is a plain Python callable:
```python
def pde_operator(feature_map, X_colloc) -> WeightedLinearSystem:
    ...
```
where `feature_map.d2(X, axis)` returns $\partial^2 H / \partial x_{\text{axis}}^2$.

In [3]:
def poisson_1d_operator(fm, X: torch.Tensor) -> WeightedLinearSystem:
    """-u'' = π² sin(πx)  ⟺  -H''β = π² sin(πx)"""
    H_xx = fm.d2(X, 0)
    rhs = (math.pi ** 2
           * torch.sin(math.pi * X[:, 0:1]).to(X.dtype))
    return WeightedLinearSystem(H=-H_xx, y=rhs, weight=1.0)

## 3. Fit CorePIELM

`CorePIELM` assembles the augmented linear system

$$\begin{pmatrix} w_{\text{pde}}\,\tilde{H} \\ w_{\text{bc}}\,H_{\text{bc}} \end{pmatrix} \beta = \begin{pmatrix} w_{\text{pde}}\,\tilde{y} \\ w_{\text{bc}}\,y_{\text{bc}} \end{pmatrix}$$

and solves for $\beta$ analytically via ridge regression in a **single pass** — no iterative optimisation.

In [4]:
model = CorePIELM(hidden_dim=300, ridge_lambda=1e-10, seed=SEED)
model.fit(dataset, pde_operator=poisson_1d_operator)
print(model)

CorePIELM(hidden_dim=300, ridge_lambda=1e-10, solver='ridge', activation='tanh')


## 4. Evaluate and Visualise

In [5]:
X_test = np.linspace(0.0, 1.0, 500).reshape(-1, 1)
u_true = np.sin(math.pi * X_test.squeeze())

with torch.no_grad():
    u_pred = model.predict(
        torch.tensor(X_test, dtype=torch.float64)
    ).numpy().squeeze()

rel_l2 = np.linalg.norm(u_true - u_pred) / (np.linalg.norm(u_true) + 1e-12)
print(f"CorePIELM  relative L² error: {rel_l2:.2e}")

fig = plot_solution_1d(
    X_test.squeeze(), u_pred, u_true,
    title=f"1D Poisson — CorePIELM  (rel-L² = {rel_l2:.1e})",
)
plt.show()

CorePIELM  relative L² error: 2.36e-08


/var/folders/mk/29j0p0jx075gv3v3bml6mtcr0000gn/T/ipykernel_51533/1032027794.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 5. Compare Model Variants

We benchmark several PyPIELM variants on the same problem and plot an accuracy vs. runtime Pareto front.

In [6]:
import time
from pypielm.models.registry import get_model

VARIANTS = [
    ("core_pielm",    dict(hidden_dim=300, ridge_lambda=1e-10)),
    ("core_pielm",    dict(hidden_dim=100, ridge_lambda=1e-8)),
    ("bayesian_pielm",dict(hidden_dim=200)),
    ("gff_pielm",     dict(hidden_dim=200)),
    ("curriculum_pielm", dict(hidden_dim=200)),
]
LABELS = ["CorePIELM-300", "CorePIELM-100", "BayesianPIELM", "GFF-PIELM", "CurriculumPIELM"]

records = []
for (name, kwargs), label in zip(VARIANTS, LABELS):
    m = get_model(name, seed=SEED, **kwargs)
    t0 = time.perf_counter()
    m.fit(dataset, pde_operator=poisson_1d_operator)
    fit_s = time.perf_counter() - t0

    with torch.no_grad():
        up = m.predict(
            torch.tensor(X_test, dtype=torch.float64)
        ).numpy().squeeze()
    rl2 = np.linalg.norm(u_true - up) / (np.linalg.norm(u_true) + 1e-12)
    print(f"  {label:25s}  fit={fit_s*1000:6.1f} ms   rel-L²={rl2:.2e}")
    records.append({"model": label, "fit_time_s": fit_s, "rel_l2": rl2})

  CorePIELM-300              fit=   7.6 ms   rel-L²=2.36e-08
  CorePIELM-100              fit=   2.9 ms   rel-L²=6.59e-07
  BayesianPIELM              fit=   6.3 ms   rel-L²=2.96e-04
  GFF-PIELM                  fit=   9.9 ms   rel-L²=1.54e-09
  CurriculumPIELM            fit=  97.1 ms   rel-L²=4.02e-07


In [7]:
fig_pareto = plot_pareto(
    records,
    x_metric="fit_time_s",
    y_metric="rel_l2",
    label_key="model",
    log_x=True,
    log_y=True,
)
plt.show()

/var/folders/mk/29j0p0jx075gv3v3bml6mtcr0000gn/T/ipykernel_51533/545250040.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 6. Save Publication-Ready Figure

In [8]:
fig_final = plot_solution_1d(
    X_test.squeeze(), u_pred, u_true,
    title="1D Poisson Equation — CorePIELM (300 neurons)",
    xlabel=r"$x$", ylabel=r"$u(x)$",
)
save_figure(fig_final, "poisson_1d_solution.pdf", dpi=300)
print("Saved poisson_1d_solution.pdf")

Saved poisson_1d_solution.pdf
